# Selección de una región petrolera para OilyGiant

OilyGiant quiere desarrollar **200 pozos nuevos**. El objetivo del proyecto es elegir
la región con el mayor beneficio promedio, siempre que el riesgo de pérdidas sea
**inferior al 2.5 %**.

Tenemos datos de tres regiones. Las columnas `f0`, `f1` y `f2` son las características;
`product` contiene las reservas en miles de barriles. `id` es un identificador y no se
utilizará para entrenar el modelo.

**Plan de trabajo:** revisar los datos, entrenar una regresión lineal por región,
calcular el beneficio de los pozos seleccionados y aplicar bootstrap para comparar
beneficios y riesgos.

Los números de las regiones corresponden a los archivos: región 0 es `geo_data_0.csv`.
Para ejecutarlo localmente, deja los tres CSV junto al notebook. En la plataforma,
si los archivos están en `/datasets/`, añade esa carpeta a las rutas de lectura.

In [1]:
# Cargamos las librerías necesarias.
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

## Carga de los datos

In [2]:
# Cargamos los datos de las tres regiones.
datos_0 = pd.read_csv('geo_data_0.csv', encoding='utf-8')
datos_1 = pd.read_csv('geo_data_1.csv', encoding='utf-8')
datos_2 = pd.read_csv('geo_data_2.csv', encoding='utf-8')

datos_regiones = [datos_0, datos_1, datos_2]

for region in range(3):
    print('Región:', region)
    print(datos_regiones[region].shape)
    print(datos_regiones[region].head())
    datos_regiones[region].info()
    print()

Región: 0
(100000, 5)
      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB

Región: 1
(100000, 5)
      id         f0         f1        f2     product
0  kBEdx -15.001348  -8.276000 -0.005876    3.179103
1  62mP7  14.272088  -3.475083  0.999183   26.953261
2  vyE1P   6.263187  -5.948386  5.001160  134.766305
3  KcrkZ -13.08119

In [3]:
# Revisamos valores ausentes y duplicados.
for region in range(3):
    datos = datos_regiones[region]
    print('Región:', region)
    print('Valores ausentes:')
    print(datos.isna().sum())
    print('Filas duplicadas:', datos.duplicated().sum())
    print('Repeticiones adicionales de id:', datos['id'].duplicated().sum())
    print('Reservas promedio:', datos['product'].mean())
    print()

Región: 0
Valores ausentes:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
Filas duplicadas: 0
Repeticiones adicionales de id: 10
Reservas promedio: 92.5

Región: 1
Valores ausentes:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
Filas duplicadas: 0
Repeticiones adicionales de id: 4
Reservas promedio: 68.825

Región: 2
Valores ausentes:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64
Filas duplicadas: 0
Repeticiones adicionales de id: 4
Reservas promedio: 95.0



In [4]:
# Observamos las filas que comparten un identificador.
for region in range(3):
    datos = datos_regiones[region]
    repetidos = datos[datos['id'].duplicated(keep=False)]
    print('Región:', region)
    print(repetidos.sort_values('id').to_string(index=False))
    print()

Región: 0
   id        f0        f1        f2    product
74z30  1.084962 -0.312358  6.990771 127.643327
74z30  0.741456  0.459229  5.153109 140.771492
A5aEY -0.180335  0.935548 -2.094773  33.020205
A5aEY -0.039949  0.156872  0.209861  89.249364
AGS9W -0.933795  0.116194 -3.655896  19.230453
AGS9W  1.454747 -0.479651  0.683380 126.370504
HZww2  0.755284  0.368511  1.863211  30.681774
HZww2  1.061194 -0.373969 10.430210 158.828695
QcMuo  0.635635 -0.473422  0.862670  64.578675
QcMuo  0.506563 -0.323775 -2.215583  75.496502
Tdehs  0.112079  0.430296  3.218993  60.964018
Tdehs  0.829407  0.298807 -0.049563  96.035308
TtcGQ  0.110711  1.022689  0.911381 101.318008
TtcGQ  0.569276 -0.104876  6.440215  85.350186
bsk9y  0.398908 -0.400253 10.122376 163.433078
bsk9y  0.378429  0.005837  0.160827 160.637302
bxg6G -0.823752  0.546319  3.630479  93.007798
bxg6G  0.411645  0.856830 -3.653440  73.604260
fiKDv  0.157341  1.028359  5.585586  95.817889
fiKDv  0.049883  0.841313  6.394613 137.346586

Re

## Preparación de los datos

Cada región tiene **100 000 filas**. No encontramos valores ausentes ni filas
completamente duplicadas, por lo que no es necesario imputar ni eliminar registros.

Encontramos 10 repeticiones adicionales de identificadores en la región 0 y 4 en cada
una de las otras regiones. Las filas tienen valores diferentes, por lo que las conservamos.
Tratamos cada fila como una observación de estos datos sintéticos. Si los identificadores
repetidos representaran mediciones del mismo pozo físico, sería necesario aclararlo
y separar los conjuntos por pozo para evitar compartir información.

Utilizaremos solamente `f0`, `f1` y `f2` para predecir `product`. Como todas las
características son numéricas, no necesitamos One-Hot Encoding. Usaremos regresión
lineal sin regularización, por lo que tampoco necesitamos escalar las variables.

## Entrenamiento del modelo

Dividimos cada región en **75 % para entrenamiento y 25 % para validación**.
Entrenamos una regresión lineal y calculamos el volumen promedio predicho y el RMSE.

El RMSE mide el error de las predicciones en miles de barriles. Cuanto menor sea,
más precisas son las predicciones en esta validación.

Colocamos estos pasos en una función para utilizarlos con las tres regiones.

In [5]:
def entrenar_modelo(datos):
    # Separamos los datos de entrenamiento y validación.
    datos_entrenamiento, datos_validacion = train_test_split(
        datos, test_size=0.25, random_state=12345
    )

    # Definimos las características y el objetivo.
    caracteristicas_entrenamiento = datos_entrenamiento[['f0', 'f1', 'f2']]
    caracteristicas_validacion = datos_validacion[['f0', 'f1', 'f2']]
    objetivo_entrenamiento = datos_entrenamiento['product']
    objetivo_validacion = datos_validacion['product']

    # Entrenamos una regresión lineal.
    modelo = LinearRegression()
    modelo.fit(caracteristicas_entrenamiento, objetivo_entrenamiento)
    predicciones = modelo.predict(caracteristicas_validacion)

    promedio = predicciones.mean()
    rmse = np.sqrt(mean_squared_error(objetivo_validacion, predicciones))

    # Guardamos juntas las reservas reales y sus predicciones.
    validacion = pd.DataFrame({
        'reservas': objetivo_validacion.to_numpy(),
        'predicciones': predicciones
    })

    return validacion, promedio, rmse

In [6]:
# Entrenamos el modelo de la región 0.
validacion_0, promedio_0, rmse_0 = entrenar_modelo(datos_0)

print('Región 0')
print('Volumen promedio predicho:', promedio_0)
print('RMSE:', rmse_0)

Región 0
Volumen promedio predicho: 92.59256778438038
RMSE: 37.5794217150813


In [7]:
# Utilizamos la misma función para la región 1.
validacion_1, promedio_1, rmse_1 = entrenar_modelo(datos_1)

print('Región 1')
print('Volumen promedio predicho:', promedio_1)
print('RMSE:', rmse_1)

Región 1
Volumen promedio predicho: 68.72854689544603
RMSE: 0.8930992867756166


In [8]:
# Utilizamos la misma función para la región 2.
validacion_2, promedio_2, rmse_2 = entrenar_modelo(datos_2)

print('Región 2')
print('Volumen promedio predicho:', promedio_2)
print('RMSE:', rmse_2)

Región 2
Volumen promedio predicho: 94.96504596800492
RMSE: 40.02970873393434


## Observaciones del entrenamiento

La región 0 obtuvo un volumen promedio predicho de **92.593** y un RMSE de **37.579**.
La región 1 obtuvo un promedio de **68.729** y un RMSE de **0.893**.
La región 2 obtuvo un promedio de **94.965** y un RMSE de **40.030**.

Aunque la región 1 tiene el menor volumen promedio, sus predicciones son mucho más
precisas. No podemos elegir una región solo por su promedio o por su RMSE: todavía
debemos comparar el beneficio y el riesgo de los pozos seleccionados.

## Preparación del cálculo de ganancias

El presupuesto es de **100 millones de USD** para desarrollar 200 pozos. Cada unidad
de `product` representa 1000 barriles y genera **4500 USD** de ingresos.

Para recuperar la inversión, los 200 pozos seleccionados deben producir un promedio
de al menos **111.111 miles de barriles por pozo**.

In [9]:
# Guardamos las condiciones del negocio en variables.
presupuesto = 100_000_000
pozos_explorados = 500
pozos_seleccionados = 200
ingreso_por_barril = 4.5
barriles_por_unidad = 1000
ingreso_por_unidad = ingreso_por_barril * barriles_por_unidad
repeticiones = 1000
limite_riesgo = 0.025

In [10]:
# Calculamos las reservas promedio necesarias para recuperar la inversión.
costo_por_pozo = presupuesto / pozos_seleccionados
reservas_minimas = costo_por_pozo / ingreso_por_unidad

print('Costo por pozo:', costo_por_pozo, 'USD')
print('Reservas promedio mínimas:', reservas_minimas, 'miles de barriles')
print()

for region in range(3):
    print('Región:', region)
    print('Reservas promedio:', datos_regiones[region]['product'].mean())

Costo por pozo: 500000.0 USD
Reservas promedio mínimas: 111.11111111111111 miles de barriles

Región: 0
Reservas promedio: 92.5
Región: 1
Reservas promedio: 68.825
Región: 2
Reservas promedio: 95.0


## Observaciones

Las reservas promedio son **92.5** en la región 0, **68.825** en la región 1 y
**95.0** en la región 2. Todas están por debajo de las reservas mínimas calculadas.

Por eso es importante seleccionar los mejores candidatos. El promedio de todos los
pozos de una región no es necesariamente el promedio de los 200 que vamos a elegir.

## Ganancia de los 200 mejores pozos

Seleccionamos por las **predicciones más altas**, pero calculamos la ganancia con
las **reservas reales de esos mismos pozos**. Así podemos evaluar los errores del modelo.

La ganancia es la suma de las reservas reales, multiplicada por 4500 USD, menos el presupuesto.

In [11]:
def calcular_ganancia(validacion, cantidad=200):
    # Seleccionamos los pozos con las predicciones más altas.
    seleccionados = validacion.sort_values('predicciones', ascending=False).head(cantidad)

    # Calculamos los ingresos con las reservas reales de esos pozos.
    reservas_totales = seleccionados['reservas'].sum()
    ganancia = reservas_totales * ingreso_por_unidad - presupuesto

    return ganancia

In [12]:
# Guardamos los 200 pozos seleccionados de cada región, incluidas sus predicciones.
validaciones = [validacion_0, validacion_1, validacion_2]
selecciones = []
ganancias_iniciales = []

for region in range(3):
    seleccionados = validaciones[region].sort_values(
        'predicciones', ascending=False
    ).head(pozos_seleccionados)
    selecciones.append(seleccionados)

    ganancia = calcular_ganancia(validaciones[region], pozos_seleccionados)
    ganancias_iniciales.append(ganancia)

    print('Región:', region)
    print('Reservas totales predichas:', seleccionados['predicciones'].sum())
    print('Reservas totales reales:', seleccionados['reservas'].sum())
    print('Ganancia:', ganancia, 'USD')
    print()

region_inicial = ganancias_iniciales.index(max(ganancias_iniciales))
print('Región con mayor ganancia inicial:', region_inicial)

Región: 0
Reservas totales predichas: 31102.330838811387
Reservas totales reales: 29601.83565142189
Ganancia: 33208260.43139851 USD

Región: 1
Reservas totales predichas: 27746.026782163477
Reservas totales reales: 27589.081548181137
Ganancia: 24150866.966815114 USD

Región: 2
Reservas totales predichas: 29603.898658318365
Reservas totales reales: 28245.22214133296
Ganancia: 27103499.635998324 USD

Región con mayor ganancia inicial: 0


## Observaciones de la selección inicial

La ganancia fue de aproximadamente **33.208 millones de USD** para la región 0,
**24.151 millones** para la región 1 y **27.103 millones** para la región 2.
La elección inicial sería la **región 0**.

Sin embargo, aquí seleccionamos los mejores 200 entre los 25 000 registros de
validación. En una exploración solamente podremos estudiar 500 candidatos.
Por eso necesitamos el bootstrap antes de tomar la decisión final.

## Cálculo de riesgos y ganancias con bootstrap

En cada repetición tomamos **500 candidatos con reemplazo** de toda la validación,
seleccionamos los **200 con las predicciones más altas** y calculamos su ganancia.
Repetimos este proceso **1000 veces por región**.

El reemplazo permite que una fila aparezca más de una vez: es parte de la simulación,
no significa que vamos a perforar el mismo pozo varias veces.

Aunque el enunciado hace referencia a los 200 mejores del paso anterior, remuestrear
solamente esos 200 dejaría fuera la exploración de 500 candidatos. Por eso utilizamos
las predicciones y reservas de **toda la validación**.

In [13]:
def hacer_bootstrap(validacion, repeticiones=1000, candidatos=500, cantidad=200):
    estado = np.random.RandomState(12345)
    ganancias = []

    for i in range(repeticiones):
        # Tomamos candidatos al azar, conservando juntas reservas y predicciones.
        muestra = validacion.sample(n=candidatos, replace=True, random_state=estado)

        # La función selecciona los mejores pozos dentro de esta muestra.
        ganancia = calcular_ganancia(muestra, cantidad)
        ganancias.append(ganancia)

    return pd.Series(ganancias)

In [14]:
# Calculamos el beneficio promedio, el intervalo del 95 % y el riesgo.
resultados = []

for region in range(3):
    ganancias = hacer_bootstrap(
        validaciones[region], repeticiones, pozos_explorados, pozos_seleccionados
    )

    ganancia_media = ganancias.mean()
    limite_inferior = ganancias.quantile(0.025)
    limite_superior = ganancias.quantile(0.975)
    riesgo = (ganancias < 0).mean()

    resultados.append([region, ganancia_media, limite_inferior, limite_superior, riesgo])

    print('Región:', region)
    print('Ganancia promedio:', ganancia_media, 'USD')
    print('Intervalo del 95 %:', limite_inferior, '-', limite_superior, 'USD')
    print('Riesgo de pérdidas:', riesgo * 100, '%')
    print()

Región: 0
Ganancia promedio: 3961649.8480237117 USD
Intervalo del 95 %: -1112155.4589049604 - 9097669.41553423 USD
Riesgo de pérdidas: 6.9 %



Región: 1
Ganancia promedio: 4560451.057866608 USD
Intervalo del 95 %: 338205.0939898458 - 8522894.538660347 USD
Riesgo de pérdidas: 1.5 %



Región: 2
Ganancia promedio: 4044038.665683568 USD
Intervalo del 95 %: -1633504.1339559986 - 9503595.749237997 USD
Riesgo de pérdidas: 7.6 %



## Observaciones del bootstrap

La región 0 tuvo una ganancia promedio de **3.962 millones de USD** y un riesgo de
pérdidas de **6.9 %**. La región 1 tuvo una ganancia promedio de **4.560 millones**
y un riesgo de **1.5 %**. La región 2 tuvo una ganancia promedio de **4.044 millones**
y un riesgo de **7.6 %**.

Los percentiles 2.5 y 97.5 delimitan el 95 % central de las ganancias simuladas.
En las regiones 0 y 2, el límite inferior es negativo. En la región 1 es positivo.
Este intervalo describe la distribución simulada de ganancias, no garantiza un
resultado concreto.

Una pérdida es una ganancia menor que cero. El riesgo se obtiene dividiendo la cantidad
de pérdidas entre las 1000 repeticiones. Luego multiplicamos por 100 para expresarlo
como porcentaje.

In [15]:
# Comparamos las regiones que cumplen el límite de riesgo.
resumen = pd.DataFrame(resultados, columns=[
    'region', 'ganancia_promedio', 'limite_inferior', 'limite_superior', 'riesgo'
])
print(resumen.to_string(index=False))
print()

# Un riesgo igual a 2.5 % tampoco cumple la condición.
regiones_validas = resumen[resumen['riesgo'] < limite_riesgo]

if len(regiones_validas) > 0:
    mejor_region = regiones_validas.sort_values('ganancia_promedio', ascending=False).iloc[0]
    print('Región recomendada:', int(mejor_region['region']))
    print('Ganancia promedio:', mejor_region['ganancia_promedio'], 'USD')
    print('Riesgo de pérdidas:', mejor_region['riesgo'] * 100, '%')
else:
    print('Ninguna región cumple el límite de riesgo.')

 region  ganancia_promedio  limite_inferior  limite_superior  riesgo
      0       3.961650e+06    -1.112155e+06     9.097669e+06   0.069
      1       4.560451e+06     3.382051e+05     8.522895e+06   0.015
      2       4.044039e+06    -1.633504e+06     9.503596e+06   0.076

Región recomendada: 1
Ganancia promedio: 4560451.057866608 USD
Riesgo de pérdidas: 1.5 %


# Conclusiones

El objetivo fue elegir una región para desarrollar 200 pozos, considerando tanto
la ganancia promedio como el riesgo de pérdidas.

## Preparación y entrenamiento

Revisamos los tres conjuntos de datos y no encontramos valores ausentes ni filas
completamente duplicadas. Conservamos las filas con identificadores repetidos porque
sus valores eran diferentes. Excluimos `id` de las características del modelo.

Dividimos cada región en **75 % para entrenamiento y 25 % para validación**.
Entrenamos solamente modelos de **regresión lineal**, como indica el enunciado.
La región 1 obtuvo el menor RMSE, aproximadamente **0.893 miles de barriles**.

## Ganancias y riesgos

Las reservas promedio de las tres regiones quedaron por debajo del punto de equilibrio
de **111.111 miles de barriles por pozo**. Esto muestra la importancia de seleccionar
los candidatos utilizando las predicciones.

La selección inicial de los 200 mejores de toda la validación favoreció a la región 0.
Sin embargo, después simulamos 1000 exploraciones de 500 candidatos y seleccionamos
los mejores 200 en cada una para representar las condiciones del negocio.

| Región | Ganancia promedio (millones USD) | Intervalo del 95 % (millones USD) | Riesgo de pérdidas |
|---|---:|---:|---:|
| 0 | 3.962 | [-1.112, 9.098] | 6.9 % |
| 1 | 4.560 | [0.338, 8.523] | 1.5 % |
| 2 | 4.044 | [-1.634, 9.504] | 7.6 % |

## Selección final

Recomiendo la **región 1 (`geo_data_1.csv`)**. Es la única que cumple el requisito de
riesgo de pérdidas inferior al 2.5 % y obtiene una ganancia promedio estimada de
**4.560 millones de USD**.

La elección final **no coincide** con la inicial. Comparar solamente los 200 mejores
de toda la validación no considera la limitación de explorar 500 candidatos ni el riesgo.

Estos resultados corresponden a datos sintéticos y a la semilla 12345. Son estimaciones,
no una garantía de ganancias. Si los identificadores repetidos pertenecieran al mismo
pozo físico, necesitaríamos aclarar su tratamiento antes de aplicar esta conclusión.